# REATS — Real-time Explainable ATR · Kaggle T4 Full Pipeline

**Data Ingestion → Multi-Viewpoint Aug → Module B Train → Module C XAI → End-to-End Demo**

| Module | Role | Target |
|--------|------|--------|
| **A** | YOLOv4 detector (CSPDarknet53 + SPP + PANet) | mAP@0.5 ≥ 75% |
| **B** | ConvNeXt_tiny × 6 ensemble classifier | Accuracy ≥ 92 %, ECE ≤ 0.05 |
| **C** | GradCAM / SHAP / MC-Dropout XAI | Faithfulness AUC ≥ 0.80 |
| **D** | Streamlit live dashboard | ≤ 40 ms/frame, ≥ 20 FPS |

**Taxonomy**: 43 military targets — AIR (25 classes) · GROUND (13) · NAVAL (6)

**Before running**: Edit → Notebook settings → Accelerator → **GPU T4 x1**

---

### Kaggle datasets to add as inputs (right panel → + Add data)

| Dataset slug | Pipeline key | Domain |
|---|---|---|
| `deepnewbie/flir-thermal-images-dataset` | `FLIR_Thermal` | thermal IR |
| `deepnewbie/flir-adas-v2` | `FLIR_ADAS_v2` | thermal IR |
| `xuanzhangyang/hit-uav` | `HIT_UAV` | aerial |
| `guofeng23/hrsc2016` | `HRSC2016` | naval |
| `rhyams/ship-detection` | `Ships_Aerial` | naval |
| `airbus/cgi-planes-in-satellite-imagery-v1` | `CGI_Planes` | air |
| `airbus/airbus-aircraft-detection` | `Airbus_Aircraft` | air |
| `javiersanchezsoriano/vehicle-dataset-for-ml` | `Vehicle_Dataset` | ground |

> Any missing dataset is silently skipped — synthetic fallback fills the gap automatically.

In [ ]:
import torch, sys, platform

print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'Platform: {platform.platform()}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    p = torch.cuda.get_device_properties(0)
    vram_gb = p.total_memory / 1e9
    print(f'\nGPU  : {p.name}')
    print(f'VRAM : {vram_gb:.1f} GB')
    BATCH_SIZE = 128 if vram_gb >= 20 else 64   # A100 → 128, T4 → 64
else:
    print('WARNING: No GPU — enable in Notebook settings → Accelerator')
    BATCH_SIZE = 16

print(f'Auto batch_size = {BATCH_SIZE}')

In [ ]:
import subprocess, sys

deps = [
    'kornia>=0.7.0',
    'grad-cam>=1.4.8',
    'shap>=0.44.0',
    'mlflow>=2.10.0',
    'torchmetrics>=1.0.0',
    'pyyaml>=6.0',
    'seaborn>=0.12.0',
    'lime>=0.2.0.1',
    'pyngrok>=7.0',
    'opencv-python-headless>=4.8.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + deps)
print('All packages installed.')

In [ ]:
import subprocess, sys, time
from pathlib import Path

REPO   = 'https://github.com/trinhvutuandat34/Real-time-Explainable-Automatic-Target-Recognition-System.git'
BRANCH = 'claude/clever-goodall-fsmqnr'
ROOT   = Path('/kaggle/working/reats')
REATS  = ROOT / 'REATS'

if ROOT.exists():
    r = subprocess.run(
        ['git', '-C', str(ROOT), 'pull', 'origin', BRANCH],
        capture_output=True, text=True,
    )
    print(r.stdout.strip() or 'Already up to date.')
else:
    subprocess.check_call(
        ['git', 'clone', '-b', BRANCH, '--depth', '1', REPO, str(ROOT)]
    )
    print(f'Cloned {BRANCH} -> {ROOT}')

for p in [str(REATS), str(ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

from config import CLASSES, NUM_CLASSES, TARGET_META
from modules.module_b_classifier import (
    build_convnext, EnsembleClassifier, build_loaders,
    train_full_pipeline, train_ensemble, load_ensemble,
    evaluate, compute_ece, KorniaAugmentPipeline,
)
from modules.module_c_xai import (
    GradCAMExplainer, MCDropoutWrapper,
    faithfulness_deletion_auc, faithfulness_insertion_auc,
    overlay_heatmap,
)
from modules.augmentation_viewpoint import MultiViewpointAugmentor
from ingestion.pipeline import IngestPipeline

print(f'REATS loaded OK | {NUM_CLASSES} classes | device={device}')
print('First 8 classes:', CLASSES[:8])

In [ ]:
from pathlib import Path

# ── Output paths ────────────────────────────────────────────────────────
DATA_DIR = Path('/kaggle/working/reats/REATS/data')
CKPT_DIR = Path('/kaggle/working/checkpoints')
RUNS_DIR = Path('/kaggle/working/mlruns')
for d in [DATA_DIR, CKPT_DIR, RUNS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Kaggle input dataset paths ──────────────────────────────────────────
# Keys MUST match entries in ingestion/label_maps.yaml exactly.
# Paths reflect where Kaggle mounts each dataset — adjust if you used
# different slugs when adding data via the right panel.
# Any path that does not exist is silently skipped; synthetic fallback
# fills the remaining shortfall automatically.
#
# Datasets added in this session (right panel → + Add data):
#   deepnewbie/flir-thermal-images-dataset          → FLIR_Thermal
#   samdazel/teledyne-flir-adas-thermal-dataset-v2  → FLIR_ADAS_v2
#   guofeng/hrsc2016                                → HRSC2016
#   andrewmvd/ship-detection                        → Ships_Aerial
#   tomluther/ships-in-google-earth                 → Ships_Google_Earth
#   siddharthkumarsah/ships-in-aerial-images        → Ships_Vessels_Aerial
#   lilitopia/swimship-wake-imagery-mass             → SWIM
#   organizations/airbusgeo/airbus-aircrafts-sample-dataset → CGI_Planes
#   kbhartiya83/swimming-pool-and-car-detection     → SwimmingPool_Car
#   alpereniek/vehicle-detection-from-satellite-images-data-set → Vehicle_Dataset
#   humansintheloop/semantic-segmentation-of-aerial-imagery → Aerial_Segmentation
#   trnhvtunt/dataset1  (HIT-UAV v1.2.1 thermal COCO JSON)  → HIT_UAV_v2
#   trnhvtunt/dataset2  (fixed_wing / rotary_wing mp4 clips) → Dataset2_Folders
#   atilol/aerialimageryforroofsegmentation                  → Aerial_Roof_Seg
DATASET_INPUTS = {
    # ── Thermal IR ──────────────────────────────────────────────────────
    'FLIR_Thermal':         Path('/kaggle/input/datasets/deepnewbie/flir-thermal-images-dataset'),
    'FLIR_ADAS_v2':         Path('/kaggle/input/datasets/samdazel/teledyne-flir-adas-thermal-dataset-v2'),
    # ── Naval ────────────────────────────────────────────────────────────
    'HRSC2016':             Path('/kaggle/input/datasets/guofeng/hrsc2016'),
    'Ships_Aerial':         Path('/kaggle/input/datasets/andrewmvd/ship-detection'),
    'Ships_Google_Earth':   Path('/kaggle/input/datasets/tomluther/ships-in-google-earth'),
    'Ships_Vessels_Aerial': Path('/kaggle/input/datasets/siddharthkumarsah/ships-in-aerial-images'),
    'SWIM':                 Path('/kaggle/input/datasets/lilitopia/swimship-wake-imagery-mass'),
    # ── Air ──────────────────────────────────────────────────────────────
    'CGI_Planes':           Path('/kaggle/input/datasets/organizations/airbusgeo/airbus-aircrafts-sample-dataset'),
    'Airbus_Aircraft':      Path('/kaggle/input/airbus-aircraft-detection'),   # standard slug (may be absent)
    # ── Ground ───────────────────────────────────────────────────────────
    'SwimmingPool_Car':     Path('/kaggle/input/datasets/kbhartiya83/swimming-pool-and-car-detection'),
    'Vehicle_Dataset':      Path('/kaggle/input/datasets/alpereniek/vehicle-detection-from-satellite-images-data-set'),
    # ── Mixed aerial ─────────────────────────────────────────────────────
    'Aerial_Segmentation':  Path('/kaggle/input/datasets/humansintheloop/semantic-segmentation-of-aerial-imagery'),
    # ── User datasets ────────────────────────────────────────────────────
    'HIT_UAV_v2':           Path('/kaggle/input/datasets/trnhvtunt/dataset1'),
    'Dataset2_Folders':     Path('/kaggle/input/datasets/trnhvtunt/dataset2'),
    'Aerial_Roof_Seg':      Path('/kaggle/input/datasets/atilol/aerialimageryforroofsegmentation'),
}

SPLIT_TARGETS = {'train': 170, 'val': 30, 'test': 200}   # per-class, per paper

# ── Module B hyperparams ────────────────────────────────────────────────
CONFIG = {
    'data_root':        str(DATA_DIR),
    'num_classes':      NUM_CLASSES,
    'img_size':         224,
    'batch_size':       BATCH_SIZE,
    'lr':               1e-4,
    'epochs':           300,
    'best_epoch_start': 225,   # checkpoint only from epoch 225, per paper
    'device':           device,
    'classes':          CLASSES,
    'warmup_epochs':    10,
    'min_lr':           1e-6,
    'weight_decay':     1e-2,
    'grad_clip':        1.0,
    'label_smoothing':  0.1,
    'ema_decay':        0.9999,
}

available_ds = [k for k, p in DATASET_INPUTS.items() if p.exists()]
print(f'Available Kaggle datasets ({len(available_ds)}/{len(DATASET_INPUTS)}): {available_ds}')
print(f'Classes: {NUM_CLASSES} | Batch: {BATCH_SIZE} | Epochs: {CONFIG["epochs"]}')

## Section 1 — Dataset

Two-stage pipeline:
1. **`IngestPipeline`** — parses all available Kaggle datasets (COCO JSON / YOLO txt / Pascal VOC XML / CSV / folder), maps source labels to the 43-class REATS taxonomy via `label_maps.yaml`, extracts 224×224 IR patches.
2. **`generate_flir_fallback.py`** — fills any remaining shortfall toward 170/30/200 images per class. Uses real FLIR ADAS bboxes if available; falls back to synthetic IR blob generation.

Both steps are idempotent — re-runs only fill the gap.

In [ ]:
datasets_available = {k: v for k, v in DATASET_INPUTS.items() if v.exists()}

if datasets_available:
    print(f'Running IngestPipeline on {len(datasets_available)} dataset(s)...')
    pipe = IngestPipeline(
        datasets      = datasets_available,
        out_root      = DATA_DIR,
        split_targets = SPLIT_TARGETS,
        seed          = 42,
    )
    written = pipe.run()
    total_written = sum(n for split in written.values() for n in split.values())
    print(f'Ingestion complete: {total_written} patches written.')
else:
    print('No Kaggle datasets found — will use synthetic fallback only.')
    print('Tip: add datasets via right panel + Add data for higher accuracy.')

In [ ]:
import subprocess

FALLBACK = REATS / 'generate_flir_fallback.py'
flir_path = DATASET_INPUTS.get('FLIR_Thermal')

cmd = ['python', str(FALLBACK), '--out', str(DATA_DIR)]
if flir_path and flir_path.exists():
    cmd += ['--flir', str(flir_path)]
    print('Fallback: using FLIR ADAS crops + synthetic for remaining shortfall...')
else:
    print('Fallback: synthetic-only (no FLIR dataset found)...')

r = subprocess.run(cmd, capture_output=True, text=True, cwd=str(REATS))
if r.stdout:
    print(r.stdout[-3000:])
if r.returncode != 0:
    print('STDERR:', r.stderr[-500:])

# Validate the final split counts
r2 = subprocess.run(
    ['python', str(REATS / 'dataset_validator.py'), '--root', str(DATA_DIR)],
    capture_output=True, text=True, cwd=str(REATS),
)
print(r2.stdout)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

# One sample per domain row, 5 columns
DOMAINS = ['AIR', 'GROUND', 'NAVAL']
COLS    = 5
domain_samples = {d: [] for d in DOMAINS}

for cls in CLASSES:
    dom = TARGET_META[cls]['domain']
    if dom in domain_samples and len(domain_samples[dom]) < COLS:
        img_dir = DATA_DIR / 'train' / cls
        imgs = sorted(img_dir.glob('*'))[:1] if img_dir.exists() else []
        if imgs:
            domain_samples[dom].append((cls, imgs[0]))

dom_color = {'AIR': '#1565C0', 'GROUND': '#2E7D32', 'NAVAL': '#00695C'}

fig, axes = plt.subplots(3, COLS, figsize=(COLS * 3, 9))
for row, dom in enumerate(DOMAINS):
    for col in range(COLS):
        ax = axes[row, col]
        if col < len(domain_samples[dom]):
            cls_id, img_path = domain_samples[dom][col]
            meta = TARGET_META[cls_id]
            try:
                ax.imshow(Image.open(img_path), cmap='inferno')
                ax.set_title(f"{meta['name']}\n[{cls_id}]", fontsize=7,
                             color=dom_color[dom], pad=2)
            except Exception:
                ax.set_title(cls_id, fontsize=7)
        else:
            ax.set_facecolor('#111')
            ax.text(0.5, 0.5, 'n/a', ha='center', va='center',
                    transform=ax.transAxes, color='gray')
        ax.axis('off')
    axes[row, 0].set_ylabel(dom, fontsize=13, fontweight='bold',
                             color=dom_color[dom], labelpad=6)

plt.suptitle('Sample IR patches by domain (inferno = simulated thermal palette)',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('/kaggle/working/sample_grid.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: sample_grid.png')

In [ ]:
import matplotlib.pyplot as plt

split_counts = {'train': [], 'val': [], 'test': []}
for split in split_counts:
    for cls in CLASSES:
        d = DATA_DIR / split / cls
        split_counts[split].append(sum(1 for _ in d.glob('*')) if d.exists() else 0)

x     = range(NUM_CLASSES)
w     = 0.28
fig, ax = plt.subplots(figsize=(max(14, NUM_CLASSES * 0.45), 5))
ax.bar([i - w for i in x], split_counts['train'], w, label='train', color='#1976D2', alpha=0.8)
ax.bar(x,                   split_counts['val'],   w, label='val',   color='#388E3C', alpha=0.8)
ax.bar([i + w for i in x], split_counts['test'],  w, label='test',  color='#F57C00', alpha=0.8)
ax.axhline(170, ls='--', color='#1976D2', lw=0.8, alpha=0.5, label='train target')
ax.axhline(200, ls='--', color='#F57C00', lw=0.8, alpha=0.5, label='test target')
ax.set_xticks(list(x))
ax.set_xticklabels(CLASSES, rotation=90, fontsize=7)
ax.set_ylabel('Images')
tr = sum(split_counts['train'])
va = sum(split_counts['val'])
te = sum(split_counts['test'])
ax.set_title(f'Dataset — {NUM_CLASSES} classes  ({tr} train / {va} val / {te} test)')
ax.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/class_distribution.png', dpi=120)
plt.show()
print('Saved: class_distribution.png')

## Section 2 — Module B: ConvNeXt Training

`train_full_pipeline` runs the complete training loop with:
- **Augmentation**: `MultiViewpointAugmentor` (5 physics-driven UAV viewpoint transforms) → `KorniaAugmentPipeline` (10 standard IR augmentations) — both wired in automatically
- **AMP** (automatic mixed precision) on CUDA
- **EMA** (exponential moving average, decay=0.9999) — used for validation/checkpointing
- **Warmup-cosine** LR scheduler (10 warm-up epochs → cosine decay to 1e-6)
- **MLflow** run logged to `/kaggle/working/mlruns/mlflow.db`
- Checkpoint saved whenever val_acc improves, from epoch 225 onward (per paper)

In [ ]:
import mlflow, time

mlflow.set_tracking_uri(f'sqlite:///{RUNS_DIR}/mlflow.db')

CKPT_SINGLE = str(CKPT_DIR / 'convnext_best.pth')

print(f'Training ConvNeXt_tiny  ({CONFIG["epochs"]} epochs, best from epoch {CONFIG["best_epoch_start"]})')
print(f'Augmentor: MultiViewpointAugmentor + KorniaAugmentPipeline')
print(f'AMP + EMA + WarmupCosine | checkpoint -> {CKPT_SINGLE}')
print()

t0 = time.time()
best_val_acc, ckpt_path = train_full_pipeline(CONFIG, seed=42, ckpt_path=CKPT_SINGLE)
elapsed_min = (time.time() - t0) / 60

if best_val_acc >= 0.92:
    status = 'PASS'
elif best_val_acc >= 0.90:
    status = 'CLOSE  -> run ensemble cell below'
else:
    status = 'FAIL   -> check data, run ensemble cell below'

print(f'Best val acc : {best_val_acc:.4f}  [{status}]  (target >= 0.92)')
print(f'Wall time    : {elapsed_min:.1f} min')

In [ ]:
# ── OPTIONAL — run if single model is below 0.92 ──────────────────────
# Trains 6 independent ConvNeXt models and averages their softmax outputs.
# Time: ~6x longer than the single-model cell.
# Each model checkpointed separately; session restarts are safe to resume.
TRAIN_ENSEMBLE = best_val_acc < 0.92

if TRAIN_ENSEMBLE:
    print(f'Single model {best_val_acc:.4f} < 0.92 — training 6-model ensemble...')
    ckpt_paths = train_ensemble(CONFIG, n_models=6, ckpt_dir=str(CKPT_DIR))
    ensemble   = load_ensemble(ckpt_paths, num_classes=NUM_CLASSES, device=device)
    ensemble.eval()
    USE_ENSEMBLE = True
    print(f'Ensemble ready: {len(ckpt_paths)} models')
else:
    print(f'Single model reached {best_val_acc:.4f} >= 0.92 — ensemble not needed.')
    USE_ENSEMBLE = False

## Section 3 — Evaluation

In [ ]:
import torch
import torch.nn as nn
from collections import defaultdict

# Load the best model (or ensemble) for evaluation
if USE_ENSEMBLE:
    eval_model = ensemble
else:
    eval_model = build_convnext(NUM_CLASSES).to(device)
    ckpt  = torch.load(CKPT_SINGLE, map_location=device)
    state = ckpt.get('ema_state_dict', ckpt.get('state_dict', ckpt))
    eval_model.load_state_dict(state)
eval_model.eval()

_, val_loader, test_loader = build_loaders(CONFIG)
criterion = nn.CrossEntropyLoss()

_, test_acc = evaluate(eval_model, test_loader, criterion, device)
ece = compute_ece(eval_model, test_loader, device, is_probs=USE_ENSEMBLE)

chk = lambda v, t, op='ge': ('PASS' if (v >= t if op == 'ge' else v <= t) else 'FAIL')

print('=' * 52)
print(f'  Test Accuracy : {test_acc:.4f}   [{chk(test_acc, 0.92)}]  target >= 0.92')
print(f'  ECE           : {ece:.5f}  [{chk(ece, 0.05, "le")}]  target <= 0.05')
print(f'  Model type    : {"Ensemble (6 models)" if USE_ENSEMBLE else "Single ConvNeXt_tiny"}')
print('=' * 52)

# Collect all predictions for downstream analysis
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        logits = eval_model(imgs.to(device))
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(labels.tolist())

# ImageFolder assigns indices alphabetically; recover name -> idx mapping
idx_to_cls  = {v: k for k, v in test_loader.dataset.class_to_idx.items()}
ordered_cls = test_loader.dataset.classes   # alphabetically sorted for cm axes

# Per-domain accuracy breakdown
dom_correct = defaultdict(int)
dom_total   = defaultdict(int)
for pred, true in zip(all_preds, all_labels):
    cls_id = idx_to_cls.get(true, '')
    dom    = TARGET_META.get(cls_id, {}).get('domain', 'UNKNOWN')
    dom_correct[dom] += int(pred == true)
    dom_total[dom]   += 1

print()
print('Per-domain accuracy:')
for dom in ['AIR', 'GROUND', 'NAVAL']:
    tot = dom_total.get(dom, 0)
    cor = dom_correct.get(dom, 0)
    acc = cor / tot if tot else 0.0
    print(f'  {dom:<8}  {acc:.4f}  ({cor}/{tot})')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(all_labels, all_preds)

cell_sz = max(0.4, 12 / NUM_CLASSES)
fig, ax = plt.subplots(figsize=(NUM_CLASSES * cell_sz + 2, NUM_CLASSES * cell_sz + 1))
sns.heatmap(
    cm,
    annot=(NUM_CLASSES <= 20),
    fmt='d',
    cmap='Blues',
    xticklabels=ordered_cls,
    yticklabels=ordered_cls,
    ax=ax,
    linewidths=0.3 if NUM_CLASSES <= 20 else 0,
)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True',      fontsize=11)
ax.set_title(
    f'Confusion Matrix — Test Set  (acc={test_acc:.3f}, {NUM_CLASSES} classes)',
    fontsize=12,
)
plt.xticks(rotation=90, fontsize=max(5, 8 - NUM_CLASSES // 10))
plt.yticks(rotation=0,  fontsize=max(5, 8 - NUM_CLASSES // 10))
plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)
plt.show()
print('Saved: confusion_matrix.png')

## Section 4 — Module C: XAI

- **Grad-CAM**: gradient-weighted class activation map — lights up the region that drove the decision
- **Faithfulness AUC**: progressively mask (deletion) or reveal (insertion) top-saliency pixels and measure class-probability drop/rise; AUC is the target metric (≥ 0.80)
- **MC Dropout**: Bayesian uncertainty estimate — high entropy = OOD / low-confidence flag

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2

# GradCAM needs a plain ConvNeXt (not the ensemble wrapper)
cam_model = ensemble.models[0] if USE_ENSEMBLE else eval_model
cam_model.eval()
gradcam = GradCAMExplainer(cam_model)

# Gather up to 3 test samples per domain
samples_by_dom = {'AIR': [], 'GROUND': [], 'NAVAL': []}
with torch.no_grad():
    for imgs, labels in test_loader:
        for img, lbl in zip(imgs, labels):
            cls_id = idx_to_cls.get(lbl.item(), '')
            dom    = TARGET_META.get(cls_id, {}).get('domain', '')
            if dom in samples_by_dom and len(samples_by_dom[dom]) < 3:
                samples_by_dom[dom].append((img, lbl.item(), cls_id))
        if all(len(v) >= 3 for v in samples_by_dom.values()):
            break

fig, axes = plt.subplots(3, 6, figsize=(18, 9))

for row, dom in enumerate(['AIR', 'GROUND', 'NAVAL']):
    for col, (img, lbl_idx, cls_id) in enumerate(samples_by_dom[dom]):
        x    = img.unsqueeze(0).to(device)
        pred = cam_model(x).argmax(1).item()
        hmap = gradcam.explain(x, pred)

        orig = (img.permute(1, 2, 0).cpu().numpy() * 0.5 + 0.5)
        gray = (orig[:, :, 0] * 255).clip(0, 255).astype(np.uint8)
        gray_rgb = np.stack([gray] * 3, -1)
        heat_rgb = cv2.applyColorMap((hmap * 255).astype(np.uint8), cv2.COLORMAP_JET)[..., ::-1]
        over = cv2.addWeighted(gray_rgb, 0.55, heat_rgb, 0.45, 0)

        ax_raw = axes[row, col * 2]
        ax_cam = axes[row, col * 2 + 1]
        ax_raw.imshow(gray, cmap='gray')
        ax_raw.set_title(f'True: {cls_id}', fontsize=8)
        ax_raw.axis('off')
        pred_name = idx_to_cls.get(pred, str(pred))
        color = 'green' if pred == lbl_idx else 'red'
        ax_cam.imshow(over)
        ax_cam.set_title(f'Pred: {pred_name}', fontsize=8, color=color)
        ax_cam.axis('off')
    axes[row, 0].set_ylabel(dom, fontsize=12, fontweight='bold', labelpad=40)

plt.suptitle('Grad-CAM  |  left: input  right: attention heatmap  (green=correct, red=wrong)',
             fontsize=11)
plt.tight_layout()
plt.savefig('/kaggle/working/gradcam_by_domain.png', dpi=150)
plt.show()
print('Saved: gradcam_by_domain.png')

In [ ]:
import numpy as np, time

faith_model = ensemble.models[0] if USE_ENSEMBLE else eval_model
faith_model.eval()

del_aucs, ins_aucs = [], []
N_FAITH = 30   # enough for stable estimate; each call takes ~0.5 s on T4

t0 = time.time()
with torch.no_grad():
    collected = 0
    for imgs, labels in test_loader:
        for img, lbl in zip(imgs[:4], labels[:4]):
            if collected >= N_FAITH:
                break
            x    = img.unsqueeze(0).to(device)
            pred = faith_model(x).argmax(1).item()
            sal  = gradcam.explain(x, pred)
            del_aucs.append(faithfulness_deletion_auc(faith_model, x, sal, pred,
                                                       steps=20, device=device))
            ins_aucs.append(faithfulness_insertion_auc(faith_model, x, sal, pred,
                                                        steps=20, device=device))
            collected += 1
        if collected >= N_FAITH:
            break

mean_del = float(np.mean(del_aucs))
mean_ins = float(np.mean(ins_aucs))
elapsed  = time.time() - t0

print(f'Faithfulness Deletion  AUC : {mean_del:.4f}  [{"PASS" if mean_del >= 0.80 else "FAIL"}]  target >= 0.80')
print(f'Faithfulness Insertion AUC : {mean_ins:.4f}  [{"PASS" if mean_ins >= 0.80 else "FAIL"}]  target >= 0.80')
print(f'n={N_FAITH} samples, {elapsed:.0f} s')

In [ ]:
# MC Dropout uncertainty: run N stochastic forward passes
mc_wrapper = MCDropoutWrapper(cam_model, n_samples=20)

imgs, labels = next(iter(test_loader))
sample = imgs[:6].to(device)

result = mc_wrapper(sample)
mean_probs  = result['mean_probs'].cpu().numpy()   # (6, C)
uncertainty = result['uncertainty'].cpu().numpy()  # (6,)  entropy

print('MC Dropout uncertainty (entropy) per sample:')
print(f'{"#":<4}  {"Class":<14}  {"Conf":>6}  {"Entropy":>8}  {"OOD?"}')
OOD_THRESH = 2.0   # heuristic: log(43) * 0.5 ~ 1.9
for i in range(6):
    pred_idx  = int(mean_probs[i].argmax())
    pred_name = ordered_cls[pred_idx] if pred_idx < len(ordered_cls) else str(pred_idx)
    conf      = mean_probs[i, pred_idx]
    ent       = uncertainty[i]
    flag      = 'OOD' if ent > OOD_THRESH else 'OK'
    print(f'{i:<4}  {pred_name:<14}  {conf:>6.3f}  {ent:>8.3f}  {flag}')

## Section 5 — End-to-End Pipeline Demo (A → B → C)

Shows the full inference pipeline on a synthetic IR frame:
1. **Module A** (`IRDetector.detect`) — YOLOv4 forward pass + NMS → bounding boxes
2. **Module B** — ConvNeXt forward pass on each ROI crop → class + confidence
3. **Module C** — Grad-CAM heatmap per detection

Module A outputs meaningful detections only after training on a labelled detection dataset; the cell below shows the correct interface regardless.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from modules.module_a_detector import IRDetector

# Synthetic 640x512 IR frame with two hot blobs (stand-ins for real FLIR video)
def make_ir_frame(h: int = 512, w: int = 640) -> np.ndarray:
    rng   = np.random.default_rng(0)
    frame = (rng.normal(loc=80, scale=10, size=(h, w))).clip(0, 255).astype(np.float32)
    for cy, cx, r, bright in [(160, 200, 30, 230), (380, 460, 20, 215)]:
        Y, X  = np.ogrid[:h, :w]
        dist  = np.sqrt((Y - cy) ** 2 + (X - cx) ** 2)
        mask  = dist < r
        frame[mask] = bright * (1 - dist[mask] / r)
    frame = frame.clip(0, 255).astype(np.uint8)
    return cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)

# Initialise Module A (untrained here — train on MSCOCO/custom FLIR for real detections)
detector   = IRDetector(num_classes=NUM_CLASSES, conf_thresh=0.25, nms_thresh=0.45)
test_frame = make_ir_frame()
detections = detector.detect(test_frame)

print(f'Module A: {len(detections)} detection(s) on synthetic frame')
for det in detections:
    cls_id = CLASSES[det['class_id']]
    dom    = TARGET_META.get(cls_id, {}).get('domain', '?')
    print(f'  {cls_id:<14}  domain={dom}  conf={det["conf"]:.3f}  bbox={det["bbox"]}')

# Visualise frame (annotated if detections exist)
vis = test_frame.copy()
for det in detections:
    x1, y1, x2, y2 = [int(v) for v in det['bbox']]
    cls_id = CLASSES[det['class_id']]
    color  = tuple(int(c) for c in TARGET_META.get(cls_id, {}).get('color_bgr', [0, 255, 0]))
    cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
    cv2.putText(vis, f'{cls_id} {det["conf"]:.2f}', (x1, y1 - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(test_frame[:, :, 0], cmap='inferno')
axes[0].set_title('Synthetic IR Frame', fontsize=11)
axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Module A — {len(detections)} detection(s)', fontsize=11)
axes[1].axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/module_a_demo.png', dpi=150)
plt.show()

In [ ]:
import time
from torchvision import transforms

roi_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])


def reats_pipeline(frame_bgr: np.ndarray, top_k: int = 3):
    """
    A -> B -> C pipeline for one IR frame.
    Returns list of result dicts and latency_ms.
    """
    t0 = time.perf_counter()

    dets = detector.detect(frame_bgr)
    if not dets:
        # Treat the whole frame as one ROI when Module A has no detections yet
        h, w = frame_bgr.shape[:2]
        dets = [{'bbox': [0, 0, w, h], 'conf': 1.0, 'class_id': 0}]

    results = []
    bm = ensemble.models[0] if USE_ENSEMBLE else eval_model
    bm.eval()
    cam = GradCAMExplainer(bm)

    for det in dets:
        x1, y1, x2, y2 = [int(v) for v in det['bbox']]
        roi = frame_bgr[max(0, y1):y2, max(0, x1):x2]
        if roi.size == 0:
            continue
        x     = roi_transform(roi).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = bm(x)
            probs  = torch.softmax(logits, dim=-1)[0].cpu().numpy()
        top_idx   = probs.argsort()[::-1][:top_k]
        pred_cls  = ordered_cls[top_idx[0]] if top_idx[0] < len(ordered_cls) else CLASSES[top_idx[0]]
        heatmap   = cam.explain(x, int(top_idx[0]))
        results.append({
            'bbox':       det['bbox'],
            'class_name': pred_cls,
            'confidence': float(probs[top_idx[0]]),
            'domain':     TARGET_META.get(pred_cls, {}).get('domain', '?'),
            'top_k':      [(ordered_cls[i] if i < len(ordered_cls) else str(i),
                            float(probs[i])) for i in top_idx],
            'heatmap':    heatmap,
            'roi':        roi,
        })

    latency_ms = (time.perf_counter() - t0) * 1000
    return results, latency_ms


# Run on the synthetic test frame
pipe_results, latency_ms = reats_pipeline(test_frame)

lat_ok = latency_ms <= 40
print(f'End-to-end latency: {latency_ms:.1f} ms  [{"PASS" if lat_ok else "OVER TARGET"}]  (target <= 40 ms)')
print()
for r in pipe_results:
    print(f'  [{r["domain"]}]  {r["class_name"]}  conf={r["confidence"]:.3f}')
    for cls_name, p in r['top_k']:
        print(f'      {cls_name:<15}  {p:.3f}')

# Visualise result panel
n_res = len(pipe_results)
if n_res > 0:
    fig, axes = plt.subplots(n_res, 3, figsize=(12, 3.5 * n_res),
                             squeeze=False)
    for i, r in enumerate(pipe_results):
        roi_gray = r['roi'][:, :, 0] if r['roi'].ndim == 3 else r['roi']
        hmap_col = cv2.applyColorMap(
            (r['heatmap'] * 255).astype(np.uint8), cv2.COLORMAP_JET
        )[..., ::-1]
        h224, w224 = 224, 224
        roi_rgb  = cv2.resize(np.stack([roi_gray] * 3, -1), (w224, h224))
        hmap_res = cv2.resize(hmap_col, (w224, h224))
        overlay  = cv2.addWeighted(roi_rgb, 0.55, hmap_res, 0.45, 0)

        bar_vals   = [p for _, p in r['top_k']]
        bar_labels = [n for n, _ in r['top_k']]

        axes[i, 0].imshow(roi_gray, cmap='gray')
        axes[i, 0].set_title('ROI crop', fontsize=9)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(overlay)
        axes[i, 1].set_title(f'Grad-CAM  [{r["class_name"]}  {r["confidence"]:.2f}]', fontsize=9)
        axes[i, 1].axis('off')

        axes[i, 2].barh(bar_labels[::-1], bar_vals[::-1], color='#1976D2')
        axes[i, 2].set_xlim(0, 1)
        axes[i, 2].set_title(f'Top-{len(bar_vals)} probs', fontsize=9)
        axes[i, 2].tick_params(labelsize=8)

    plt.suptitle(f'A->B->C Pipeline  ({latency_ms:.0f} ms)', fontsize=12)
    plt.tight_layout()
    plt.savefig('/kaggle/working/pipeline_demo.png', dpi=150)
    plt.show()
    print('Saved: pipeline_demo.png')

## Section 6 — Module D: Streamlit Dashboard

The dashboard has 4 tabs:
- **Live Analysis** — upload a frame or FLIR video; full A→B→C pipeline; annotated output with threat-level colour overlay
- **Batch** — run on a folder of frames; export results CSV
- **Calibration** — temperature scaling; ECE + reliability diagram
- **About** — 43-class taxonomy grouped by domain + threat level legend

To serve it from Kaggle, expose port 8501 via an ngrok tunnel.

In [ ]:
import subprocess, sys

# ── Set your ngrok token (free account at https://ngrok.com) ───────────
NGROK_TOKEN = ''   # <- paste token here to enable live dashboard

if NGROK_TOKEN:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN

    dashboard_path = REATS / 'modules' / 'module_d_dashboard.py'
    proc = subprocess.Popen([
        sys.executable, '-m', 'streamlit', 'run', str(dashboard_path),
        '--server.port', '8501',
        '--server.headless', 'true',
        '--server.fileWatcherType', 'none',
    ])
    tunnel = ngrok.connect(8501, 'http')
    print(f'Dashboard: {tunnel.public_url}')
    print('Open the URL above in a new browser tab.')
else:
    print('Set NGROK_TOKEN above to launch the live dashboard.')
    print()
    print('Alternatively, run locally after downloading the checkpoint:')
    print(f'  cd <local-clone>/REATS')
    print('  pip install streamlit kornia grad-cam shap mlflow')
    print('  streamlit run modules/module_d_dashboard.py')

In [ ]:
from pathlib import Path

print('=' * 58)
print('  REATS Artifacts — /kaggle/working/')
print('=' * 58)
ARTIFACT_EXTS = {'.pth', '.png', '.yaml', '.db', '.csv'}
for p in sorted(Path('/kaggle/working').rglob('*')):
    if p.is_file() and p.suffix in ARTIFACT_EXTS:
        size_kb = p.stat().st_size / 1024
        rel = str(p.relative_to('/kaggle/working'))
        print(f'  {rel:<50}  {size_kb:7.1f} KB')
print('=' * 58)
print()
print('IMPORTANT: /kaggle/working/ is wiped when the session ends.')
print('Download these before closing:')
print('  checkpoints/convnext_best.pth         <- single model')
print('  checkpoints/convnext_{0..5}.pth       <- ensemble (if trained)')
print()
print('Performance summary:')
print(f'  Accuracy      {test_acc:.4f}  [{"PASS" if test_acc >= 0.92 else "FAIL"}]  target >= 0.92')
print(f'  ECE           {ece:.5f}  [{"PASS" if ece <= 0.05 else "FAIL"}]  target <= 0.05')
if 'mean_del' in dir():
    print(f'  Faith. AUC    {mean_del:.4f}  [{"PASS" if mean_del >= 0.80 else "FAIL"}]  target >= 0.80')
if 'latency_ms' in dir():
    print(f'  Latency       {latency_ms:.1f} ms  [{"PASS" if latency_ms <= 40 else "OVER"}]  target <= 40 ms')